## **Prediksi Resiko Stunting pada Balita berdasarkan Faktor Risiko Kesehatan**

In [2]:
# Cell 1: Import Library & Menggabungkan Data

import pandas as pd
import numpy as np

# 1. Membaca dan menggabungkan data dari 4 sheet
file_path = 'Overall data.xlsx'
print("Membaca file Excel...")
all_sheets = pd.read_excel(file_path, sheet_name=None)
df = pd.concat(all_sheets.values(), ignore_index=True)

print(f"Data siap! Total baris: {df.shape[0]}, Total kolom: {df.shape[1]}")

Membaca file Excel...
Data siap! Total baris: 40071, Total kolom: 17


In [3]:
# Cell 2: Inspeksi "Penyakit" Data

# 1. Mengecek jumlah baris yang duplikat persis
duplicate_count = df.duplicated().sum()
print(f"Jumlah baris duplikat: {duplicate_count}")

# 2. Mengecek jumlah sel kosong (Missing Values) di setiap kolom
print("\n=== JUMLAH DATA KOSONG PER KOLOM ===")
missing_data = df.isnull().sum()
print(missing_data[missing_data > 0]) # Hanya menampilkan kolom yang ada kosongnya

# 3. Melihat tipe data saat ini
print("\n=== TIPE DATA KOLOM ===")
print(df.dtypes)

Jumlah baris duplikat: 0

=== JUMLAH DATA KOSONG PER KOLOM ===
Posyandu (Integrated Service Post)    354
Age (Month)                             2
Weight for Height                       5
dtype: int64

=== TIPE DATA KOLOM ===
No.                                     int64
Gender                                    str
Date of Birth                          object
Regency/City                              str
District                                  str
Public Health Center (Puskesmas)          str
Posyandu (Integrated Service Post)        str
Date of Measurement                    object
Age (Month)                           float64
Weight                                 object
Height                                float64
Weight for Age                            str
Z-Score  W/A                           object
Height for Age                            str
Z-Score H/A                            object
Weight for Height                         str
Z-Score W/H                          

In [4]:
# Cell 3: Konversi Tipe Data & Penanganan Missing Values

# 1. Menghapus kolom 'No.' karena hanya nomor urut (noise untuk model ML)
if 'No.' in df.columns:
    df.drop(columns=['No.'], inplace=True)

# 2. Mengobati penyakit Tipe Data: Memaksa teks menjadi angka numerik
# Parameter errors='coerce' sangat sakti: jika ada teks aneh (misal "12 kg" atau salah ketik), 
# pandas akan otomatis mengubahnya menjadi NaN (kosong) agar tidak error.
kolom_bermasalah = ['Weight', 'Z-Score  W/A', 'Z-Score H/A', 'Z-Score W/H']
for col in kolom_bermasalah:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Menangani Missing Values berdasarkan temuan di Cell 2 (ditambah efek dari coerce di atas)
# a. Kolom Numerik (Angka): Isi dengan nilai tengah (Median)
kolom_numerik_kosong = ['Age (Month)', 'Weight', 'Z-Score  W/A', 'Z-Score H/A', 'Z-Score W/H']
for col in kolom_numerik_kosong:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# b. Kolom Teks (String): Isi dengan data yang paling sering muncul (Modus)
kolom_teks_kosong = ['Posyandu (Integrated Service Post)', 'Weight for Height']
for col in kolom_teks_kosong:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()[0])

print("=== STATUS SETELAH PEMBERSIHAN ===")
print(f"Total sel kosong sekarang: {df.isnull().sum().sum()}")
print("\n=== CEK ULANG TIPE DATA YANG DIPERBAIKI ===")
print(df[['Weight', 'Z-Score  W/A', 'Z-Score H/A', 'Z-Score W/H']].dtypes)

=== STATUS SETELAH PEMBERSIHAN ===
Total sel kosong sekarang: 0

=== CEK ULANG TIPE DATA YANG DIPERBAIKI ===
Weight          float64
Z-Score  W/A    float64
Z-Score H/A     float64
Z-Score W/H     float64
dtype: object


In [5]:
# Cell 4: Konversi Datetime & Inspeksi Nilai Unik

# 1. Konversi kolom tanggal menjadi tipe Datetime yang dipahami oleh Python
kolom_tanggal = ['Date of Birth', 'Date of Measurement']
print("Sedang mengonversi format tanggal...")
for col in kolom_tanggal:
    if col in df.columns:
        # errors='coerce' sangat penting agar teks aneh tidak membuat program error, melainkan jadi NaT (Not a Time)
        df[col] = pd.to_datetime(df[col], errors='coerce')

# 2. Mengecek apakah ada tanggal yang rusak setelah dikonversi (menjadi NaT)
print("\n=== JUMLAH TANGGAL YANG RUSAK (NaT) ===")
print(df[kolom_tanggal].isnull().sum())

# 3. Mengecek nilai unik pada kolom kategorikal sebelum kita ubah jadi angka (Encoding)
kolom_kategorikal = ['Gender', 'Weight for Age', 'Height for Age', 'Weight for Height']
print("\n=== NILAI UNIK KOLOM KATEGORIKAL ===")
for col in kolom_kategorikal:
    if col in df.columns:
        # unique() akan menampilkan semua variasi teks yang pernah diketik di kolom tersebut
        nilai_unik = df[col].unique()
        print(f"-> {col}: {nilai_unik}")

Sedang mengonversi format tanggal...

=== JUMLAH TANGGAL YANG RUSAK (NaT) ===
Date of Birth          1116
Date of Measurement    8322
dtype: int64

=== NILAI UNIK KOLOM KATEGORIKAL ===
-> Gender: <StringArray>
['F', 'M', 'P', 'L']
Length: 4, dtype: str
-> Weight for Age: <StringArray>
['Normal', 'Underfed', 'Malnutrition', 'Overnutrition']
Length: 4, dtype: str
-> Height for Age: <StringArray>
['Stunted', 'Not Stunted']
Length: 2, dtype: str
-> Weight for Height: <StringArray>
['Normal', 'Thin', 'Obese', 'Very Thin']
Length: 4, dtype: str


C:\Users\Administrator\AppData\Local\Temp\ipykernel_18668\677172238.py:9: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors='coerce')


In [6]:
# Cell 5: Fix Format Tanggal & Encoding Kategorikal

# 1. MEMPERBAIKI FORMAT TANGGAL (Pelajaran dari Warning)
kolom_tanggal = ['Date of Birth', 'Date of Measurement']
print("Memperbaiki format tanggal (Hari/Bulan/Tahun)...")
for col in kolom_tanggal:
    if col in df.columns:
        # Menambahkan parameter dayfirst=True untuk format tanggal Indonesia
        df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

# Cek kembali jumlah tanggal yang rusak setelah diperbaiki
print("\n=== SISA TANGGAL RUSAK (NaT) SETELAH DIPERBAIKI ===")
print(df[kolom_tanggal].isnull().sum())


# 2. ENCODING: MENGUBAH TEKS JADI ANGKA
print("\n=== MELAKUKAN ENCODING DATA KATEGORIKAL ===")

# a. Encoding Gender (Menyatukan Bahasa Inggris & Indonesia)
# F/P (Perempuan) = 1, M/L (Laki-laki) = 0
gender_map = {'F': 1, 'P': 1, 'M': 0, 'L': 0}
if 'Gender' in df.columns:
    df['Gender'] = df['Gender'].map(gender_map)

# b. Encoding Target: Height for Age (Stunting)
# Not Stunted = 0, Stunted = 1 (Karena kita ingin mendeteksi stunting)
hfa_map = {'Not Stunted': 0, 'Stunted': 1}
if 'Height for Age' in df.columns:
    df['Height for Age'] = df['Height for Age'].map(hfa_map)

# c. Encoding Status Gizi Lainnya (Ordinal Encoding: makin parah makin tinggi angkanya)
wa_map = {'Normal': 0, 'Underfed': 1, 'Malnutrition': 2, 'Overnutrition': 3}
if 'Weight for Age' in df.columns:
    df['Weight for Age'] = df['Weight for Age'].map(wa_map)

wh_map = {'Normal': 0, 'Thin': 1, 'Very Thin': 2, 'Obese': 3}
if 'Weight for Height' in df.columns:
    df['Weight for Height'] = df['Weight for Height'].map(wh_map)

print("Proses Encoding selesai!")
print("\n=== CEK 5 BARIS PERTAMA SETELAH ENCODING ===")
display(df[['Gender', 'Height for Age', 'Weight for Age', 'Weight for Height']].head())

Memperbaiki format tanggal (Hari/Bulan/Tahun)...

=== SISA TANGGAL RUSAK (NaT) SETELAH DIPERBAIKI ===
Date of Birth          1116
Date of Measurement    8322
dtype: int64

=== MELAKUKAN ENCODING DATA KATEGORIKAL ===
Proses Encoding selesai!

=== CEK 5 BARIS PERTAMA SETELAH ENCODING ===


,Gender,Height for Age,Weight for Age,Weight for Height
0,1,1,0,0
1,0,1,0,0
2,0,1,0,0
3,0,1,0,0
4,1,1,0,0


In [7]:
# Cell 5.5: Investigasi Data Rusak (Opsional tapi Sangat Disarankan)

# 1. Memfilter dan mengambil HANYA baris yang 'Date of Measurement'-nya NaT
data_rusak = df[df['Date of Measurement'].isnull()]

print(f"Total baris dengan tanggal pengukuran rusak: {len(data_rusak)}")

# 2. Mari kita lihat wujud 5 baris pertamanya secara utuh
print("\n=== WUJUD 5 BARIS DATA YANG RUSAK TANGGALNYA ===")
display(data_rusak.head())

# 3. Investigasi Pola: Apakah kerusakan ini didominasi oleh Puskesmas tertentu?
print("\n=== DISTRIBUSI DATA RUSAK BERDASARKAN PUSKESMAS ===")
# value_counts() akan menghitung jumlah kemunculan tiap nama Puskesmas di dalam data yang rusak
print(data_rusak['Public Health Center (Puskesmas)'].value_counts().head(10))

Total baris dengan tanggal pengukuran rusak: 8322

=== WUJUD 5 BARIS DATA YANG RUSAK TANGGALNYA ===


,Gender,Date of Birth,Regency/City,District,Public Health Center (Puskesmas),Posyandu (Integrated Service Post),Date of Measurement,Age (Month),Weight,Height,Weight for Age,Z-Score W/A,Height for Age,Z-Score H/A,Weight for Height,Z-Score W/H
9,0,2016-10-29,JENEPONTO,BANGKALA,BANGKALA,BATU NAPARAKALOANG,NaT,57.0,10.0,81.0,2,-4.12,1,-5.82,0,-0.50
10,0,2017-06-19,JENEPONTO,BATANG,TOGO TOGO,BONTOSUNGGU,NaT,49.0,11.5,90.0,1,-2.93,1,-3.52,0,-1.28
11,1,2017-02-24,JENEPONTO,BATANG,TOGO TOGO,ULUGALUNG,NaT,54.0,13.1,97.0,1,-2.18,1,-2.35,0,-1.17
12,1,2018-03-24,JENEPONTO,BATANG,TOGO TOGO,ULUGALUNG,NaT,41.0,11.0,86.0,1,-2.35,1,-3.27,0,-0.47
13,0,2018-04-05,JENEPONTO,BATANG,TOGO TOGO,BONTOJANNANG,NaT,40.0,11.1,89.0,1,-2.19,1,-2.23,0,-1.27



=== DISTRIBUSI DATA RUSAK BERDASARKAN PUSKESMAS ===
Public Health Center (Puskesmas)
TAMALATEA       1324
BONTOMATENE     1070
BARANA           795
BANGKALA         609
TOLO             551
BULULOE          522
BONTORAMBA       421
BINAMU           395
TAROWANG         392
EMBO             333
Name: count, dtype: int64


In [8]:
# Cell 6 (REVISI): Membuang Kolom Redundan & Membuat Fitur BMI

# 1. MENGHAPUS KOLOM TANGGAL (Karena Umur sudah mewakili waktu)
print("=== MENGHAPUS KOLOM TANGGAL YANG REDUNDAN ===")
kolom_buang = ['Date of Birth', 'Date of Measurement']
# Kita hapus kolomnya (axis=1), bukan barisnya
df.drop(columns=[col for col in kolom_buang if col in df.columns], inplace=True)

# 2. FEATURE ENGINEERING: MENCIPTAKAN FITUR BMI
print("\n=== MEMBUAT FITUR BARU: BMI ===")

# Rumus BMI: Berat (kg) / (Tinggi (m))^2
if 'Weight' in df.columns and 'Height' in df.columns:
    tinggi_meter = df['Height'] / 100
    df['BMI'] = df['Weight'] / (tinggi_meter ** 2)
    
    # Membulatkan nilai BMI menjadi 2 angka di belakang koma agar rapi
    df['BMI'] = df['BMI'].round(2)

print("Fitur BMI berhasil dibuat!")
print(f"Total data bersih kita bertahan di: {df.shape[0]} baris!")

# 3. CEK HASIL AKHIR DATASET KITA SEBELUM MASUK KE MACHINE LEARNING
print("\n=== 5 BARIS PERTAMA DENGAN FITUR BARU ===")
# Kita tampilkan Umur, Tinggi, Berat, Target Stunting, dan fitur baru kita (BMI)
display(df[['Age (Month)', 'Height', 'Weight', 'BMI', 'Height for Age']].head())

=== MENGHAPUS KOLOM TANGGAL YANG REDUNDAN ===

=== MEMBUAT FITUR BARU: BMI ===
Fitur BMI berhasil dibuat!
Total data bersih kita bertahan di: 40071 baris!

=== 5 BARIS PERTAMA DENGAN FITUR BARU ===


,Age (Month),Height,Weight,BMI,Height for Age
0,54.0,97.5,13.2,13.89,1
1,44.0,92.0,12.0,14.18,1
2,57.0,97.0,14.0,14.88,1
3,26.0,79.0,11.0,17.63,1
4,59.0,98.0,14.6,15.20,1


In [9]:
# Cell 7: Memisahkan Fitur (X) dan Target (y) + Train-Test Split

# Import library untuk memecah data
from sklearn.model_selection import train_test_split

print("=== TAHAP 1: MEMISAHKAN FITUR (X) DAN TARGET (y) ===")
# Kita ambil fitur murni tanpa unsur kebocoran data (Data Leakage)
fitur = ['Gender', 'Age (Month)', 'Weight', 'Height', 'BMI']
X = df[fitur]
y = df['Height for Age']

print(f"Fitur yang digunakan: {X.columns.tolist()}")

print("\n=== TAHAP 2: TRAIN-TEST SPLIT ===")
# Memecah data: 80% untuk Train (Belajar), 20% untuk Test (Ujian)
# stratify=y sangat penting untuk memastikan proporsi anak stunting & normal terbagi rata di kedua set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Total Data Latih (Train): {X_train.shape[0]} baris")
print(f"Total Data Uji (Test): {X_test.shape[0]} baris")

print("\n=== TAHAP 3: CEK KESEIMBANGAN KELAS TARGET (Pada Data Latih) ===")
# value_counts() akan menghitung ada berapa anak normal (0) dan stunting (1)
distribusi_kelas = y_train.value_counts()
print(distribusi_kelas)

# Mencegah error jika nama kelasnya berbeda
if 1 in distribusi_kelas:
    persentase_stunting = (distribusi_kelas[1] / len(y_train)) * 100
    print(f"\nPersentase anak Stunting (Kelas 1) di data latih: {persentase_stunting:.2f}%")
else:
    print("\nPeringatan: Kelas 1 (Stunting) tidak ditemukan, cek kembali proses Encoding.")

=== TAHAP 1: MEMISAHKAN FITUR (X) DAN TARGET (y) ===
Fitur yang digunakan: ['Gender', 'Age (Month)', 'Weight', 'Height', 'BMI']

=== TAHAP 2: TRAIN-TEST SPLIT ===
Total Data Latih (Train): 32056 baris
Total Data Uji (Test): 8015 baris

=== TAHAP 3: CEK KESEIMBANGAN KELAS TARGET (Pada Data Latih) ===
Height for Age
0    19093
1    12963
Name: count, dtype: int64

Persentase anak Stunting (Kelas 1) di data latih: 40.44%


In [10]:
# Cell 8: Feature Scaling (Standardisasi)

# Import library untuk standardisasi
from sklearn.preprocessing import StandardScaler

print("=== TAHAP 4: FEATURE SCALING (STANDARDISASI) ===")

# 1. Inisialisasi Scaler
scaler = StandardScaler()

# 2. Proses Scaling (Aturan Emas: Fit HANYA pada Data Latih!)
# fit_transform: Komputer belajar nilai rata-rata dari X_train, lalu mengubah angkanya
X_train_scaled = scaler.fit_transform(X_train)

# transform: Komputer HANYA mengubah angka X_test menggunakan aturan dari X_train
# (Jika kita melakukan fit pada X_test, itu namanya "nyontek" atau Data Leakage!)
X_test_scaled = scaler.transform(X_test)

print("Scaling berhasil dilakukan!")

# Mari kita lihat perbedaannya agar Anda paham wujud datanya
print("\n=== PERBANDINGAN WUJUD DATA (Baris Pertama) ===")
print(f"Nama Fitur    : {X_train.columns.tolist()}")
print(f"Sebelum Scale : {X_train.iloc[0].values}")
# Dibulatkan 3 angka di belakang koma agar rapi
print(f"Setelah Scale : {[round(val, 3) for val in X_train_scaled[0]]}")

=== TAHAP 4: FEATURE SCALING (STANDARDISASI) ===
Scaling berhasil dilakukan!

=== PERBANDINGAN WUJUD DATA (Baris Pertama) ===
Nama Fitur    : ['Gender', 'Age (Month)', 'Weight', 'Height', 'BMI']
Sebelum Scale : [  0.    53.    11.   102.5   10.47]
Setelah Scale : [np.float64(-0.906), np.float64(1.264), np.float64(-0.01), np.float64(1.395), np.float64(-0.058)]


In [11]:
# Cell 9: Melatih dan Mengadu Algoritma (Grand Finale)

# Import library Algoritma Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# Import library untuk metrik evaluasi
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

print("=== TAHAP 5 & 6: TRAINING DAN EVALUASI MODEL ===")
print("Sedang melatih 3 algoritma sekaligus. Harap tunggu sebentar...\n")

# 1. Inisialisasi ketiga model ("Otak" yang masih kosong)
# random_state=42 pada Random Forest agar hasil acakannya konsisten
models = {
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Naïve Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(random_state=42)
}

# Tempat menyimpan hasil rapor masing-masing algoritma
rapor_model = []

# 2. Proses Belajar (Training) dan Ujian (Testing)
for nama_model, model in models.items():
    # a. TRAINING (Belajar dari data latih yang sudah di-scale)
    model.fit(X_train_scaled, y_train)
    
    # b. TESTING (Menebak soal ujian dari data test yang sudah di-scale)
    y_pred = model.predict(X_test_scaled)
    
    # c. PENILAIAN (Membandingkan tebakan dengan kunci jawaban y_test)
    akurasi = accuracy_score(y_test, y_pred)
    presisi = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Memasukkan nilai ke dalam rapor (dalam bentuk persentase)
    rapor_model.append({
        "Algoritma": nama_model,
        "Accuracy (%)": round(akurasi * 100, 2),
        "Precision (%)": round(presisi * 100, 2),
        "Recall (%)": round(recall * 100, 2),
        "F1-Score (%)": round(f1 * 100, 2)
    })

# 3. Menampilkan Hasil Komparasi
df_rapor = pd.DataFrame(rapor_model)
print("=== HASIL KOMPARASI ALGORITMA ===")
display(df_rapor.sort_values(by="Accuracy (%)", ascending=False).reset_index(drop=True))

=== TAHAP 5 & 6: TRAINING DAN EVALUASI MODEL ===
Sedang melatih 3 algoritma sekaligus. Harap tunggu sebentar...

=== HASIL KOMPARASI ALGORITMA ===


,Algoritma,Accuracy (%),Precision (%),Recall (%),F1-Score (%)
0,Random Forest,97.31,97.05,96.27,96.65
1,K-Nearest Neighbors,96.24,96.26,94.38,95.31
2,Naïve Bayes,42.55,41.20,98.52,58.10


In [12]:
# Cell 10: Menguji Generalisasi dengan 10-Fold Cross-Validation

from sklearn.model_selection import cross_val_score
import numpy as np

print("=== TAHAP 7: UJI GENERALISASI (10-FOLD CROSS-VALIDATION) ===")
print("Sedang membagi data menjadi 10 lipatan dan menguji model 10 kali...")
print("Ini mungkin memakan waktu beberapa detik...\n")

# 1. Kita panggil kembali model juara kita (Random Forest)
model_juara = RandomForestClassifier(random_state=42)

# 2. Eksekusi 10-Fold Cross Validation pada data latih
# cv=10 artinya membagi data jadi 10 bagian. 
# n_jobs=-1 menyuruh Python menggunakan semua inti prosesor komputer Anda agar lebih cepat.
skor_ujian = cross_val_score(model_juara, X_train_scaled, y_train, cv=10, scoring='accuracy', n_jobs=-1)

# 3. Menampilkan Hasil
print("=== NILAI RAPOR DARI 10 PUTARAN UJIAN ===")
# Kita ubah formatnya jadi persentase agar mudah dibaca
skor_persen = np.round(skor_ujian * 100, 2)
for i, skor in enumerate(skor_persen, 1):
    print(f"Putaran ke-{i}: {skor}%")

# 4. Menghitung Rata-rata dan Tingkat Kestabilan (Standar Deviasi)
rata_rata = skor_ujian.mean() * 100
standar_deviasi = skor_ujian.std() * 100

print("\n=== KESIMPULAN GENERALISASI ===")
print(f"Rata-rata Akurasi : {rata_rata:.2f}%")
print(f"Tingkat Goyah (Std Dev) : ±{standar_deviasi:.2f}%")

if standar_deviasi < 2.0:
    print("-> STATUS: SANGAT STABIL! Model memiliki tingkat generalisasi yang luar biasa.")
else:
    print("-> STATUS: KURANG STABIL. Model mungkin mengalami sedikit overfitting di beberapa bagian data.")

=== TAHAP 7: UJI GENERALISASI (10-FOLD CROSS-VALIDATION) ===
Sedang membagi data menjadi 10 lipatan dan menguji model 10 kali...
Ini mungkin memakan waktu beberapa detik...

=== NILAI RAPOR DARI 10 PUTARAN UJIAN ===
Putaran ke-1: 97.38%
Putaran ke-2: 97.54%
Putaran ke-3: 97.04%
Putaran ke-4: 96.91%
Putaran ke-5: 97.26%
Putaran ke-6: 97.26%
Putaran ke-7: 97.63%
Putaran ke-8: 97.97%
Putaran ke-9: 97.57%
Putaran ke-10: 97.63%

=== KESIMPULAN GENERALISASI ===
Rata-rata Akurasi : 97.42%
Tingkat Goyah (Std Dev) : ±0.30%
-> STATUS: SANGAT STABIL! Model memiliki tingkat generalisasi yang luar biasa.


In [13]:
# Cell 11: Fine Tuning (Mencari Setelan Terbaik)

from sklearn.model_selection import GridSearchCV

print("=== TAHAP 8: FINE TUNING (MENCARI SETELAN TERBAIK) ===")
print("Laptop akan menguji berbagai kombinasi. Harap bersabar...\n")

# 1. Menentukan daftar kombinasi tuas (Hyperparameter Grid)
# Kita buat sangat ringan agar laptop aman!
param_grid = {
    'n_estimators': [50, 100],           # Tuas 1: Coba bangun 50 pohon, lalu coba 100 pohon
    'max_depth': [10, 20, None],         # Tuas 2: Batasi kedalaman pohon (10, 20, atau biarkan bebas/None)
    'min_samples_split': [2, 5]          # Tuas 3: Syarat minimal data untuk memecah cabang
}
# Total ada 2 x 3 x 2 = 12 kombinasi setelan yang akan dicoba.

# 2. Inisialisasi Pencari Otomatis (GridSearchCV)
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,                 # Ujian 3 lipatan (Cross-Validation)
    scoring='accuracy',
    n_jobs=1,             # Gunakan 1 inti CPU agar aman dari crash
    verbose=2             # Menampilkan progres di layar
)

# 3. Mulai pencarian! (12 kombinasi x 3 ujian = 36 kali melatih model)
# Peringatan: Ini mungkin memakan waktu 1-3 menit.
grid_search.fit(X_train_scaled, y_train)

# 4. Menampilkan Hasil Setelan Terbaik
print("\n=== HASIL PENCARIAN FINE TUNING ===")
print(f"Setelan Tuas (Parameter) Terbaik : {grid_search.best_params_}")
print(f"Akurasi Rata-rata Saat Tuning    : {round(grid_search.best_score_ * 100, 2)}%")

# 5. Uji Akhir: Bawa "Otak" dengan setelan terbaik ini untuk menebak Soal Ujian (X_test)
model_final_terbaik = grid_search.best_estimator_
akurasi_final_test = model_final_terbaik.score(X_test_scaled, y_test)

print(f"\n=> AKURASI FINAL DI DATA UJI (TEST SET): {round(akurasi_final_test * 100, 2)}%")

=== TAHAP 8: FINE TUNING (MENCARI SETELAN TERBAIK) ===
Laptop akan menguji berbagai kombinasi. Harap bersabar...

Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END .max_depth=10, min_samples_split=2, n_estimators=50; total time=   0.6s
[CV] END .max_depth=10, min_samples_split=2, n_estimators=50; total time=   0.6s
[CV] END .max_depth=10, min_samples_split=2, n_estimators=50; total time=   0.6s
[CV] END max_depth=10, min_samples_split=2, n_estimators=100; total time=   1.3s
[CV] END max_depth=10, min_samples_split=2, n_estimators=100; total time=   1.3s
[CV] END max_depth=10, min_samples_split=2, n_estimators=100; total time=   1.3s
[CV] END .max_depth=10, min_samples_split=5, n_estimators=50; total time=   0.6s
[CV] END .max_depth=10, min_samples_split=5, n_estimators=50; total time=   0.5s
[CV] END .max_depth=10, min_samples_split=5, n_estimators=50; total time=   0.6s
[CV] END max_depth=10, min_samples_split=5, n_estimators=100; total time=   1.2s
[CV] END max_de

In [14]:
# Cell 12: Mengekspor Model dan Scaler (Tahap Akhir Notebook)

import joblib
import os

print("=== TAHAP 9: MENYIMPAN MODEL UNTUK DEPLOYMENT ===")

# 1. Menentukan nama file
file_model = 'model_rf_stunting_terbaik.pkl'
file_scaler = 'scaler_stunting.pkl'

# 2. Menyimpan model final terbaik dari Grid Search
joblib.dump(grid_search.best_estimator_, file_model)

# 3. Menyimpan Scaler yang kita buat di Cell 8
joblib.dump(scaler, file_scaler)

# Cek apakah file benar-benar terbuat
if os.path.exists(file_model) and os.path.exists(file_scaler):
    print(f"SUKSES! File '{file_model}' dan '{file_scaler}' telah berhasil dibuat.")
    print("Silakan cek folder proyek/direktori Jupyter Notebook Anda.")
    print("Kedua file inilah yang akan Anda bawa ke dalam kode aplikasi Web Anda (Flask/Streamlit)!")
else:
    print("Gagal menyimpan file. Cek kembali direktori Anda.")

=== TAHAP 9: MENYIMPAN MODEL UNTUK DEPLOYMENT ===
SUKSES! File 'model_rf_stunting_terbaik.pkl' dan 'scaler_stunting.pkl' telah berhasil dibuat.
Silakan cek folder proyek/direktori Jupyter Notebook Anda.
Kedua file inilah yang akan Anda bawa ke dalam kode aplikasi Web Anda (Flask/Streamlit)!
